# Homework 12: Optimal Feedback Control
**Computational Sensorimotor Control**

You build a complete OFC controller and test it with a **target-jump paradigm**.
Two target conditions run throughout:

- **Point target:** the hand must reach a specific point. Both x and y position matter.
- **Bar target:** the hand must reach a vertical bar (5 cm tall) at the same x-distance. Only x matters; y is free within ±2.5 cm.

A perpendicular target jump (1 cm) occurs mid-movement. For the **point target**, this jump
is task-relevant and must be corrected. For the **bar target**, the jump is task-irrelevant —
the hand is still heading toward the bar regardless. The controller should ignore it.

**Structure:** You build each component step by step, then test it.

| Part | Component | Fill-in |
|------|-----------|---------|
| 1 | Plant (linearize arm) | 2 lines |
| 2 | Cost function (point vs bar Q) | 2 lines |
| 3 | Controller (Riccati backward) | 3 lines |
| 4 | Estimator (Kalman filter) | 3 lines |
| 5 | LQG simulation (baseline) | — |
| 6 | Target jump (implement) | 2 lines |
| 7 | Jump at three times | — |
| 8 | Jump × MIP (point vs bar) | — |
| 9 | Sensory degradation | 2 lines |

---
## Part 0: Setup

In [ ]:
# Install the smc package (run this cell first)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.lines as mlines
plt.rcParams.update({
    'figure.figsize': (10, 5), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'
})

from smc import Arm, Q_REF, mass_matrix
arm = Arm()
fk = arm.forward_kinematics
ik = arm.inverse_kinematics

# Colors (matching lecture)
C_MJ = '#27AE60'; C_LQR = '#2E86AB'; C_LQG = '#E74C3C'
C_EPH = '#8E44AD'; C_FOV = '#F39C12'

# ── Reach setup ──
start_hand = fk(Q_REF)
target_xy  = start_hand + np.array([0.12, 0.0])  # 12 cm rightward
q_target   = ik(target_xy[0], target_xy[1])
dt = 0.01; T = 0.5; N = int(T / dt)
x0 = np.zeros(4); x0[:2] = Q_REF - q_target
t_ms  = np.arange(N) * dt * 1000
tf_ms = np.arange(N + 1) * dt * 1000

# Reach and perpendicular directions (for later)
reach_dir = (target_xy - start_hand)
reach_dir = reach_dir / np.linalg.norm(reach_dir)
perp_dir  = np.array([-reach_dir[1], reach_dir[0]])

print(f'Reach: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) → '
      f'({target_xy[0]*100:.1f}, {target_xy[1]*100:.1f}) cm')

---
## Part 1: Linearize the Arm

LQR requires linear dynamics: x_{t+1} = Ax_t + Bu_t.

**State:** x = [δq₁, δq₂, δq̇₁, δq̇₂] (4×1) — deviations from target  
**Input:** u = [τ₁, τ₂] (2×1) — joint torques

At the target posture, Coriolis terms vanish and the continuous-time dynamics are:
- A_c = [[0, I], [0, 0]] — the arm coasts
- B_c = [[0], [M⁻¹]] — torques act through the inverse mass matrix

Discretize at dt = 10 ms: A = I + A_c·dt, B = B_c·dt

In [ ]:
def linearize_arm(q_lin, dt):
    """Linearize 2-link arm at (q_lin, 0). Returns A (4×4), B (4×2)."""
    Ac = np.zeros((4, 4))
    Ac[0, 2] = 1.0; Ac[1, 3] = 1.0   # velocity drives position
    Bc = np.zeros((4, 2))
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (2 lines): Compute M⁻¹ and place it in Bc
    #   Hint: mass_matrix(q_lin) returns M; use np.linalg.inv()
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    return np.eye(4) + Ac * dt, Bc * dt

A, B = linearize_arm(q_target, dt)
print(f'A shape: {A.shape}'); print(f'B shape: {B.shape}')

# Helper: task-space output matrices
def task_C(q):
    J = arm.jacobian(q); C = np.zeros((2,4)); C[:,:2] = J; return C

def task_C_vel(q):
    J = arm.jacobian(q); Cv = np.zeros((4,4)); Cv[:2,:2] = J; Cv[2:,2:] = J; return Cv

C_out = task_C(q_target)
print(f'\nC (maps 4D state → 2D hand position): shape {C_out.shape}')

---
## Part 2: Cost Function — Point Target vs Bar Target

The cost J = Σ xᵀQ_t x + Σ uᵀRu penalizes state errors and torque.

**Point target:** Q_N penalizes both x and y hand-position errors equally (w = 100).  
**Bar target:** Q_N penalizes x errors heavily (w = 100) and y errors weakly (w = 5).

R is the same for both — the effort cost doesn't depend on the target shape.

**Your task:** Build the bar target Q_N. The point target Q is provided.

In [ ]:
# Control cost (same for both targets)
r = 0.0001
R_mat = r * np.eye(2)

# ── Point target Q ──
def build_Q_point(q_lin, N, w_pos=100.0, w_vel=1.0):
    C = task_C(q_lin)
    Q_N = w_pos * (C.T @ C) + np.diag([0, 0, w_vel, w_vel])
    return [np.zeros((4,4)) if t < N else Q_N for t in range(N+1)]

Q_point = build_Q_point(q_target, N)

# ── Bar target Q ──
C_x = C_out[0:1, :]   # (1×4) maps state → hand x-position
C_y = C_out[1:2, :]   # (1×4) maps state → hand y-position

# ════════════════════════════════════════════════════════════════
# ▶ FILL IN (2 lines): Build Q_N for the bar target
#   Penalize x heavily (w=100) and y weakly (w=5), plus velocity (w=1)
#   Hint: C_x.T @ C_x penalizes only x; C_y.T @ C_y penalizes only y
# ════════════════════════════════════════════════════════════════
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

print(f'Point Q_N rank: {np.linalg.matrix_rank(Q_point[N])}')
print(f'Bar   Q_N rank: {np.linalg.matrix_rank(Q_bar[N])}')

---
## Part 3: Backward Riccati Recursion

Solve for the optimal time-varying gains L_t by running the Riccati equation
backward from the endpoint:

- S_N = Q_N (terminal cost-to-go)
- L_t = (R + BᵀS_{t+1}B)⁻¹ BᵀS_{t+1}A (optimal gain)
- S_t = Q_t + AᵀS_{t+1}(A − BL_t) (propagate cost-to-go backward)

**Your task:** Fill in the three core lines of the recursion.

In [ ]:
def riccati_backward(A, B, Q_seq, R, N):
    """Backward Riccati recursion. Returns L (N gains), S (N+1 cost-to-go matrices)."""
    S = [None] * (N + 1)
    L = [None] * N
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (3 lines):
    #   1. Set terminal condition S[N]
    #   2. Compute L[t] from S[t+1] (inside the loop)
    #   3. Compute S[t] from Q_seq[t], S[t+1], L[t] (inside the loop)
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    for t in range(N - 1, -1, -1):
        BtS = B.T @ S[t + 1]
        ...  # YOUR CODE HERE
        ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    return L, S

# Solve for both targets
L_point, S_point = riccati_backward(A, B, Q_point, R_mat, N)
L_bar,   S_bar   = riccati_backward(A, B, Q_bar,   R_mat, N)

# ── Figure: Compare gain profiles ──
fig, ax = plt.subplots(figsize=(8, 4))
gn_pt = [np.linalg.norm(L_point[t]) for t in range(N)]
gn_br = [np.linalg.norm(L_bar[t])   for t in range(N)]
ax.plot(t_ms, gn_pt, color=C_LQR, lw=2, label='Point target')
ax.plot(t_ms, gn_br, color=C_LQG, lw=2, label='Bar target')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('‖L_t‖')
ax.set_title('Riccati gains: point vs bar (plotted forward, computed backward)')
ax.legend(); plt.tight_layout(); plt.show()

print('Both gain profiles are computed BEFORE the movement starts.')
print('They differ because the cost function Q differs — not because of noise or sensors.')

**Q3a:** Why do the point and bar gain profiles differ, even though the arm and dynamics are identical?  
**Q3b:** The gains are plotted forward in time (0 = start, 500 = end) but computed backward. Why does ‖L_t‖ peak just before the endpoint and then drop at the very last timestep?

---
## Part 4: Observation Model and Kalman Filter

The controller acts on the *estimated* state x̂_t, not the true state. The Kalman filter
fuses noisy sensory signals to produce this estimate.

**Observation model:** y_t = Hx_t + v_t (8×1 measurement)
- Top 4 rows of H = I₄ (proprioception measures joint state directly)
- Bottom 4 rows of H = C_vel (vision measures hand pos+vel via Jacobian)

**Key point:** H is identical for both point and bar targets. The Kalman filter
does not know or care about the task. Only V_t (noise) changes over time as the
hand enters foveal range (separation principle).

In [ ]:
def build_obs_model(q_lin):
    Cv = task_C_vel(q_lin)
    H = np.zeros((8,4)); H[:4,:] = np.eye(4); H[4:,:] = Cv
    V_per = np.diag([np.radians(1)**2]*2 + [np.radians(5)**2]*2 +  # proprioception
                    [0.005**2]*2 + [0.01**2]*2)                      # peripheral vision
    V_fov = np.diag([np.radians(1)**2]*2 + [np.radians(5)**2]*2 +  # proprioception unchanged
                    [0.001**2]*2 + [0.005**2]*2)                     # foveal vision
    return H, V_per, V_fov

def sensory_schedule(N, frac=0.70, width=0.08):
    return np.array([1/(1+np.exp(-(t/N - frac)/width)) for t in range(N)])

H_obs, V_per, V_fov = build_obs_model(q_target)
alpha = sensory_schedule(N)
W = np.diag([1e-6, 1e-6, 5e-4, 5e-4])  # process noise

print(f'H: {H_obs.shape} — same for point and bar (separation principle)')

In [ ]:
def kalman_step(xh, P, u, y, A, B, H, W, V):
    """Single KF predict-update. Returns x̂_new (4×1), P_new (4×4), K (4×8)."""
    n = A.shape[0]
    # Predict (provided — uses efference copy, 0 ms delay)
    x_pred = A @ xh + B @ u
    P_pred = A @ P @ A.T + W
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (3 lines): Kalman gain, corrected estimate, updated covariance
    #   Hint: K = P_pred Hᵀ (H P_pred Hᵀ + V)⁻¹
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    return x_new, P_new, K

---
## Part 5: Baseline LQG Reach — Point vs Bar (No Jump)

Before testing target jumps, run the baseline: 200 trials of the 12 cm reach
with both target conditions. The Kalman filter is **identical** in both — only L_t differs.

This should reproduce the minimum intervention result from the lecture:
the bar target produces an elongated endpoint distribution along y.

In [ ]:
def simulate_lqg(A, B, H, L, W, V_per, V_fov, alpha, x0, N, rng=None, n_trials=1):
    n, p = 4, H.shape[0]
    if rng is None: rng = np.random.default_rng(42)
    Wc = np.linalg.cholesky(W + 1e-14*np.eye(n))
    x_all = np.zeros((n_trials, N+1, n))
    for tr in range(n_trials):
        x = x0.copy(); xh = np.zeros(n); P = 0.05*np.eye(n)
        x_all[tr,0] = x
        for t in range(N):
            u = -L[t] @ xh
            x = A @ x + B @ u + Wc @ rng.standard_normal(n)
            x_all[tr,t+1] = x
            V_t = (1-alpha[t])*V_per + alpha[t]*V_fov
            Vc = np.linalg.cholesky(V_t + 1e-14*np.eye(p))
            y = H @ x + Vc @ rng.standard_normal(p)
            xh, P, _ = kalman_step(xh, P, u, y, A, B, H, W, V_t)
    return x_all

def states_to_hand(x_seq, q_ref):
    return np.array([fk(q_ref + x_seq[i,:2])*100 for i in range(len(x_seq))])

# Run baseline for both targets
n_trials = 200
x_pt, x_br = [simulate_lqg(A,B,H_obs,L,W,V_per,V_fov,alpha,x0,N,
               rng=np.random.default_rng(42),n_trials=n_trials)
               for L in [L_point, L_bar]]

hands_pt = np.zeros((n_trials, N+1, 2))
hands_br = np.zeros((n_trials, N+1, 2))
for tr in range(n_trials):
    hands_pt[tr] = states_to_hand(x_pt[tr], q_target)
    hands_br[tr] = states_to_hand(x_br[tr], q_target)
h_pt_mean = np.mean(hands_pt, axis=0)
h_br_mean = np.mean(hands_br, axis=0)
bar_center = target_xy * 100

# ── Figure: Baseline point vs bar ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for tr in range(25):
    ax.plot(hands_pt[tr,:,0], hands_pt[tr,:,1], color=C_LQR, alpha=0.1, lw=0.8)
    ax.plot(hands_br[tr,:,0], hands_br[tr,:,1], color=C_LQG, alpha=0.1, lw=0.8)
ax.plot([bar_center[0]]*2, [bar_center[1]-2.5, bar_center[1]+2.5],
        color=C_LQG, lw=4, alpha=0.4, solid_capstyle='round')
ax.plot(bar_center[0], bar_center[1], 'kx', ms=10, mew=2)
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.legend(handles=[mlines.Line2D([],[],color=C_LQR,lw=2,label='Point target'),
                   mlines.Line2D([],[],color=C_LQG,lw=2,label='Bar target')], fontsize=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Baseline reach — point vs bar')
# Expand y
ay = np.concatenate([hands_pt[:,:,1].ravel(), hands_br[:,:,1].ravel()])
ym = np.mean([ay.min(), ay.max()]); ys = max(ay.max()-ay.min(), 2)
ax.set_ylim(ym-ys*0.7, ym+ys*0.7)

ax = axes[1]
ep_pt = hands_pt[:,-1,:]; ep_br = hands_br[:,-1,:]
ax.scatter(ep_pt[:,0], ep_pt[:,1], c=C_LQR, alpha=0.4, s=20, label='Point')
ax.scatter(ep_br[:,0], ep_br[:,1], c=C_LQG, alpha=0.4, s=20, label='Bar')
ax.plot(bar_center[0], bar_center[1], 'kx', ms=12, mew=2)
ax.plot([bar_center[0]]*2, [bar_center[1]-2.5, bar_center[1]+2.5],
        color=C_LQG, lw=4, alpha=0.3, solid_capstyle='round')
for ep, col in [(ep_pt, C_LQR), (ep_br, C_LQG)]:
    cov = np.cov(ep.T); ev, evec = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(evec[1,1], evec[0,1]))
    ax.add_patch(Ellipse(xy=ep.mean(0), width=4*np.sqrt(max(ev[1],1e-10)),
                         height=4*np.sqrt(max(ev[0],1e-10)), angle=ang,
                         fc='none', ec=col, lw=2))
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_aspect('equal')
ax.set_title('B: Endpoint distributions (±2 SD)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'Point: along-reach var = {np.var((ep_pt-ep_pt.mean(0)) @ reach_dir):.3f}, '
      f'perp var = {np.var((ep_pt-ep_pt.mean(0)) @ perp_dir):.3f} cm²')
print(f'Bar:   along-reach var = {np.var((ep_br-ep_br.mean(0)) @ reach_dir):.3f}, '
      f'perp var = {np.var((ep_br-ep_br.mean(0)) @ perp_dir):.3f} cm²')

**Q5a:** Why is the bar target ellipse elongated along y but tight in x?  
**Q5b:** The Kalman filter is identical for both conditions. What differs, and why?

---
## Part 6: Implement the Target Jump

During the reach, the target jumps 1 cm **perpendicular** to the reach direction.
The jump stays within foveal range.

**Key concept:** A target jump of Δp in task space changes the state deviation:
the target moved, so the hand is now further from where it needs to be.
In joint space: Δx = −J⁻¹ · Δp (negative because the target moved away).

**Your task:** Convert the 1 cm perpendicular jump to joint space.

In [ ]:
def simulate_lqg_jump(A, B, H, L, W, V_per, V_fov, alpha, x0, N,
                       jump_t, jump_dq, rng):
    """Single LQG trial with target jump (state perturbation) at jump_t."""
    n, p = 4, H.shape[0]
    Wc = np.linalg.cholesky(W + 1e-14*np.eye(n))
    x = x0.copy(); xh = np.zeros(n); P = 0.05*np.eye(n)
    x_hist = np.zeros((N+1, n)); x_hist[0] = x
    for t in range(N):
        if t == jump_t:
            x[:2] += jump_dq   # target moved → deviation from target increased
        u = -L[t] @ xh
        x = A @ x + B @ u + Wc @ rng.standard_normal(n)
        x_hist[t+1] = x
        V_t = (1-alpha[t])*V_per + alpha[t]*V_fov
        Vc = np.linalg.cholesky(V_t + 1e-14*np.eye(p))
        y = H @ x + Vc @ rng.standard_normal(p)
        xh, P, _ = kalman_step(xh, P, u, y, A, B, H, W, V_t)
    return x_hist

# Convert perpendicular jump to joint space
J_inv = np.linalg.inv(arm.jacobian(q_target))
jump_cm = 0.01   # 1 cm in meters

# ════════════════════════════════════════════════════════════════
# ▶ FILL IN (2 lines): Convert the perpendicular target jump to joint space
#   1. Compute the jump vector in task space (perp_dir × jump_cm)
#   2. Convert to joint space using −J⁻¹ (negative because target moved away)
# ════════════════════════════════════════════════════════════════
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

print(f'Jump in task space: {jump_task*100} cm')
print(f'Jump in joint space: {np.degrees(jump_dq)} deg')

---
## Part 7: Target Jump at Three Times (Point Target)

Test the point-target controller with the perpendicular jump at three times:
early (100 ms), middle (250 ms), late (400 ms).

**Prediction:** Early jumps give the controller time to correct fully.
Late jumps occur when L_t is large but time is running out — endpoint error increases.

In [ ]:
jump_times = {'Early (100ms)': int(0.10/dt),
              'Middle (250ms)': int(0.25/dt),
              'Late (400ms)': int(0.40/dt)}
jump_colors = {'Early (100ms)': C_MJ, 'Middle (250ms)': C_LQR, 'Late (400ms)': C_LQG}
n_jump = 50
target_new = target_xy + perp_dir * jump_cm  # actual target after jump

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: Hand paths
ax = axes[0]
ax.plot(h_pt_mean[:,0], h_pt_mean[:,1], 'k-', lw=2, label='No jump')
for label, jt in jump_times.items():
    for trial in range(20):
        xh = simulate_lqg_jump(A,B,H_obs,L_point,W,V_per,V_fov,alpha,
                                x0,N,jt,jump_dq,np.random.default_rng(2000+trial))
        h = states_to_hand(xh, q_target)
        ax.plot(h[:,0], h[:,1], color=jump_colors[label], alpha=0.1, lw=0.8)
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=10, mew=2)
ax.legend(handles=[mlines.Line2D([],[],color=c,lw=2,label=l)
          for l,c in jump_colors.items()] +
          [mlines.Line2D([],[],color='k',lw=2,label='No jump')], fontsize=8)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Point target — jump at 3 times')

# Panel B: Endpoint error
ax = axes[1]
for label, jt in jump_times.items():
    errs = []
    for trial in range(n_jump):
        xh = simulate_lqg_jump(A,B,H_obs,L_point,W,V_per,V_fov,alpha,
                                x0,N,jt,jump_dq,np.random.default_rng(2000+trial))
        h = states_to_hand(xh, q_target)
        errs.append(np.linalg.norm(h[-1]/100 - target_new) * 100)
    ax.bar(label.split('(')[0], np.mean(errs), color=jump_colors[label], alpha=0.7)
    ax.errorbar(label.split('(')[0], np.mean(errs), yerr=np.std(errs), color='k', capsize=5)
ax.set_ylabel('Endpoint error from new target (cm)')
ax.set_title('B: Later jumps → larger error')

# Panel C: Remaining control authority
ax = axes[2]
L_norm = [np.linalg.norm(L_point[t]) for t in range(N)]
remaining = np.array([np.sum(L_norm[t:]) for t in range(N)])
ax.plot(t_ms, remaining / remaining[0], color=C_LQR, lw=2.5)
for label, jt in jump_times.items():
    ax.axvline(jt*dt*1000, color=jump_colors[label], ls='--', lw=1.5)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Remaining ΣL (normalized)')
ax.set_title('C: Control authority remaining')
plt.tight_layout(); plt.show()

**Q7a:** Why does a late jump produce a larger endpoint error, even though L_t is largest near the end?  
**Q7b:** What does Panel C tell you about the controller's ability to correct at each jump time?  
**Q7c:** If you extended the movement to 700 ms, how would the endpoint errors change?

---
## Part 8: Target Jump × Minimum Intervention

The strongest test of MIP: **same perpendicular jump at 250 ms, two target conditions.**

- **Point target:** the jump displaces the target away from the hand's path → task-relevant → correct
- **Bar target:** the jump moves the target along the bar → the hand is still heading toward the bar → task-irrelevant → ignore

This is the core prediction: identical physical event, different Q, different response.

In [ ]:
jt = int(0.25 / dt)  # jump at 250 ms

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Hand paths — point vs bar after same jump
ax = axes[0]
all_y_mip = []
for trial in range(25):
    for L_cond, col, seed_off in [(L_point, C_LQR, 3000), (L_bar, C_LQG, 3000)]:
        rng_j = np.random.default_rng(seed_off + trial)
        xh = simulate_lqg_jump(A,B,H_obs,L_cond,W,V_per,V_fov,alpha,
                                x0,N,jt,jump_dq,rng_j)
        h = states_to_hand(xh, q_target)
        ax.plot(h[:,0], h[:,1], color=col, alpha=0.12, lw=0.8)
        all_y_mip.extend(h[:,1].tolist())
ax.plot(h_pt_mean[:,0], h_pt_mean[:,1], 'k-', lw=2)
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(bar_center[0], bar_center[1], 'kx', ms=10, mew=2)
ax.plot([bar_center[0]]*2, [bar_center[1]-2.5, bar_center[1]+2.5],
        color=C_LQG, lw=4, alpha=0.3, solid_capstyle='round')
ax.legend(handles=[mlines.Line2D([],[],color=C_LQR,lw=2,label='Point + jump'),
                   mlines.Line2D([],[],color=C_LQG,lw=2,label='Bar + jump'),
                   mlines.Line2D([],[],color='k',lw=2,label='No jump')], fontsize=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Same jump, different Q — MIP in action')
ym = np.mean([min(all_y_mip), max(all_y_mip)])
ys = max(max(all_y_mip)-min(all_y_mip), 2)
ax.set_ylim(ym-ys*0.7, ym+ys*0.7)

# Panel B: Perpendicular deviation over time
ax = axes[1]
for label, L_cond, col in [('Point', L_point, C_LQR), ('Bar', L_bar, C_LQG)]:
    perp_dev = np.zeros((n_jump, N+1))
    for trial in range(n_jump):
        rng_j = np.random.default_rng(3000 + trial)
        xh = simulate_lqg_jump(A,B,H_obs,L_cond,W,V_per,V_fov,alpha,
                                x0,N,jt,jump_dq,rng_j)
        h = states_to_hand(xh, q_target)
        perp_dev[trial] = (h - h_pt_mean) @ perp_dir
    ax.plot(tf_ms, np.abs(np.mean(perp_dev, axis=0)), color=col, lw=2, label=f'{label} target')
    ax.fill_between(tf_ms, 0, np.abs(np.mean(perp_dev, axis=0)), color=col, alpha=0.1)
ax.axvline(250, color='gray', ls=':', lw=1, label='Jump')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('|Perpendicular deviation| (cm)')
ax.set_title('B: Point corrects, bar ignores'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Q8a:** Why does the bar-target controller ignore the perpendicular jump?  
**Q8b:** In the Riccati framework, what determines whether a deviation triggers correction? (Hint: Δv* = ΔxᵀS_tΔx. What is S_t for the bar target in the perpendicular direction?)  
**Q8c:** If the jump were *along* the reach direction instead of perpendicular, would both controllers correct it? Why?

---
## Part 9: Sensory Degradation × Target Jump

Muscle vibration (~80 Hz) disrupts spindle signals. We model this by inflating
proprioceptive noise 5× (variance 25×).

**Key prediction from the separation principle:** L_t is unchanged (the controller
doesn't depend on sensors). Only the Kalman filter adapts — it downweights
proprioception and relies more on vision.

**Your task:** Create the degraded V matrices and compare target-jump correction.

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN (2 lines): Create degraded proprioception noise matrices
#   Copy V_per and V_fov, then inflate the first 4 diagonal entries by 25×
# ════════════════════════════════════════════════════════════════
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

# Compare: normal vs vibration, point target, jump at each time
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Hand paths (jump at 250ms)
ax = axes[0]
for trial in range(20):
    for V_p, V_f, col in [(V_per, V_fov, C_LQR), (V_per_deg, V_fov_deg, C_LQG)]:
        rng_j = np.random.default_rng(4000+trial)
        xh = simulate_lqg_jump(A,B,H_obs,L_point,W,V_p,V_f,alpha,
                                x0,N,jt,jump_dq,rng_j)
        h = states_to_hand(xh, q_target)
        ax.plot(h[:,0], h[:,1], color=col, alpha=0.1, lw=0.8)
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=10, mew=2)
ax.legend(handles=[mlines.Line2D([],[],color=C_LQR,lw=2,label='Normal'),
                   mlines.Line2D([],[],color=C_LQG,lw=2,label='Vibration')], fontsize=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Jump at 250ms — normal vs vibration')

# Panel B: Endpoint error for each jump time × condition
ax = axes[1]
width = 0.18
for i, (label, jt_i) in enumerate(jump_times.items()):
    for j, (cond, V_p, V_f, col) in enumerate([
            ('Normal', V_per, V_fov, C_LQR),
            ('Vibration', V_per_deg, V_fov_deg, C_LQG)]):
        errs = []
        for trial in range(n_jump):
            xh = simulate_lqg_jump(A,B,H_obs,L_point,W,V_p,V_f,alpha,
                                    x0,N,jt_i,jump_dq,np.random.default_rng(5000+trial))
            h = states_to_hand(xh, q_target)
            errs.append(np.linalg.norm(h[-1]/100 - target_new) * 100)
        ax.bar(i + (j-0.5)*width, np.mean(errs), width, color=col, alpha=0.7)
        ax.errorbar(i + (j-0.5)*width, np.mean(errs), yerr=np.std(errs), color='k', capsize=4)
ax.set_xticks(range(len(jump_times)))
ax.set_xticklabels([k.split('(')[0] for k in jump_times.keys()])
ax.legend(handles=[mlines.Line2D([],[],color=C_LQR,lw=8,alpha=0.7,label='Normal'),
                   mlines.Line2D([],[],color=C_LQG,lw=8,alpha=0.7,label='Vibration')], fontsize=9)
ax.set_ylabel('Endpoint error (cm)'); ax.set_title('B: Vibration effect by jump time')
plt.tight_layout(); plt.show()

**Q9a:** Why is L_t unchanged between normal and vibration conditions? (Name the principle.)  
**Q9b:** For late jumps (400 ms), the correction should be relatively preserved despite vibration. Why? (Hint: which sensory channel dominates at 400 ms?)  
**Q9c:** For early jumps (100 ms), the correction is more degraded. Why?  
**Q9d:** If you could choose to vibrate the arm OR occlude foveal vision, which would hurt the late-jump correction more? Why?